In [2]:
import numpy as np
import pandas as pd

# ----------------------------
# File paths
# ----------------------------
main_file = r"C:\Exoplanet\159_planets_all_columns_with_AH.csv"
bh_file   = r"C:\Users\ahuss\Downloads\results_2026_02_24__16_53.csv"
# ----------------------------
# Load datasets
# ----------------------------
df_main = pd.read_csv(main_file)
df_bh   = pd.read_csv(bh_file)

# ----------------------------
# Clean planet names (strip spaces, unify case)
# ----------------------------
df_main['pl_name_clean'] = df_main['pl_name'].str.strip().str.upper()
df_bh['File_Name_clean'] = df_bh['File_Name'].str.strip().str.upper()

# ----------------------------
# Inner merge (only planets present in both CSVs)
# ----------------------------
df_merged = pd.merge(
    df_bh,
    df_main[['pl_name_clean', 'pl_trandur']],
    left_on='File_Name_clean',
    right_on='pl_name_clean',
    how='inner'
).drop(columns=['pl_name_clean', 'File_Name_clean'])

# ----------------------------
# Check for missing transit durations (should be none after inner merge)
# ----------------------------
missing = df_merged[df_merged["pl_trandur"].isna()]
if not missing.empty:
    print("These planets have no transit duration match:")
    print(missing["File_Name"])

# ----------------------------
# Compute required transits and observing time
# ----------------------------
target_error = 0.5  # desired measurement error

# Number of transits required
df_merged["N_required"] = np.ceil(
    (df_merged["B_H_Error"] / target_error) ** 2
).astype(int)

# Observing time per transit (2 × transit duration)
df_merged["time_per_transit"] = df_merged["pl_trandur"] * 2

# Total observing time in hours
df_merged["total_time_hours"] = df_merged["time_per_transit"] * df_merged["N_required"]

# ----------------------------
# Summary table
# ----------------------------
summary_cols = [
    "File_Name",
    "pl_trandur",
    "B_H_Error",
    "N_required",
    "total_time_hours"
]
print(df_merged[summary_cols])

# ----------------------------
# Total observing time
# ----------------------------
total_hours = df_merged["total_time_hours"].sum()
print("\nTotal observing time (hours):", total_hours)
print("Total observing time (days):", total_hours / 24)

# ----------------------------
# Optional: sort by efficiency (smallest time per B_H_Error first)
# ----------------------------
df_merged["efficiency"] = df_merged["B_H_Error"] / df_merged["total_time_hours"]
df_merged_sorted = df_merged.sort_values(by="efficiency", ascending=False)
print("\nTop 10 most efficient targets:")
print(df_merged_sorted[summary_cols + ["efficiency"]].head(10))

      File_Name  pl_trandur  B_H_Error  N_required  total_time_hours
0     GJ 1214 b    0.869660   0.183651           1          1.739320
1     GJ 3090 b    1.281000   0.694420           2          5.124000
2   HD 191939 d    5.360000   0.409558           1         10.720000
3      K2-138 d    2.700000   3.083066          39        210.600000
4      K2-138 f    3.200000   2.750869          31        198.400000
5       K2-18 b    2.682000   1.409811           8         42.912000
6     L 98-59 d    0.840000   0.291177           1          1.680000
7    LHS 1140 b    2.150000   0.638470           2          8.600000
8    TOI-1064 c    2.370000   1.765109          13         61.620000
9    TOI-1136 c    3.488032   0.591559           2         13.952128
10   TOI-1260 b    2.060000   1.410038           8         32.960000
11   TOI-1422 b    4.460000   0.656866           2         17.840000
12   TOI-1468 c    1.860000   0.604452           2          7.440000
13   TOI-1758 b    3.672700   1.76